# EXP-2026-008 / Q5-E — PREP P3: source-match equivalence differential

**이 노트북은 미실행 상태로 커밋된다** — 저장소 사본은 출력이 전부 비어 있고
`execution_count` 가 모두 `null` 이다. 실행 승인이 나기 전까지는 그대로 둔다.

**실행 승인은 아직 없다.** 이 노트북은 P3 *구현* 이 무엇을 할지 보여줄 뿐이고,
등록 `data.py` 를 열지 않는다. 열려면 두 장벽이 **둘 다** 열려야 한다.

| 장벽 | 지금 값 | 여는 방법 |
|---|---|---|
| `P3.OPEN_REGISTERED_DATA` | `False` | 호출 지점에서 명시적으로 opt-in |
| `P3.EXECUTION_APPROVAL_RECORD['granted']` | `False` | 별도 승인 PR 이 한 필드를 바꾼다 |

Q5-E 실행 토큰과 P1/P2 PREP 토큰은 **이름으로 거부** 된다. 다른 단계의 승인이
이 단계를 열 수 없다.

## 무엇을 확인하는가

`Q5E.match_peaks_to_annotations()` 는 산문에서 옮겨 적은 **후보 어댑터** 다.
같은 오독을 두 번 하면 "일치" 가 나오므로, oracle 은 규칙을 다시 옮겨 적은
두 번째 구현이 **아니다** — digest 로 검증한 등록 `data.py` 자체를 합성
의존성 주입 아래에서 실행하고, 그 실행 자취에서 결정을 기계적으로 읽는다.

비교 대상: peak↔annotation 매핑 · kept row 집합과 **순서** · 소비된 주석과
**소비 시점** · 반환(release) 여부 · 미매칭 주석 · 미매칭 peak · AAMI 선택
전/후 · 경계컷 전/후.

**22/22 record count 재현은 증명이 아니다.** 이 노트북은 실제 record count 를
열지 않으며, 여러 어댑터를 돌려 좋은 것을 고르는 기능도 없다.


In [1]:
# 1. DESIGN — 실행 순서를 먼저 보여준다. 이 셀은 아무것도 열지 않는다.
STAGES = [
    '1. DESIGN                              이 표와 경계 선언',
    '2. ENVIRONMENT                         repo/모듈 로드와 staleness guard',
    '3. SYNTHETIC_FIXTURES                  합성 자산만으로 회귀 스위트 실행',
    '4. DEPENDENCY_AND_APPROVAL_PREFLIGHT   의존성·장벽 점검(자격증명 이전)',
    '5. SOURCE_FILE_IDENTITY                file id → inventory → 바이트 SHA-256',
    '6. SOURCE_ORACLE_DIFFERENTIAL          등록 build_record 실행 ↔ 어댑터',
    '7. RESULT_GATE                         등록 gate 로 후보 구조 검사(등록 아님)',
    '8. BUNDLE_REPORT                       외부 동결용 digest 와 다음 단계',
]
for line in STAGES:
    print(line)
print()
print('승인 전에는 1~4 까지만 진행된다. 5 이후는 두 장벽이 모두 열려야 한다.')
print('이 노트북은 SOURCE_MATCH_ORACLE_RECORD 에 아무것도 쓰지 않는다 —')
print('등록은 Codex 인수 뒤 별도 registration PR 의 몫이다.')


1. DESIGN                              이 표와 경계 선언
2. ENVIRONMENT                         repo/모듈 로드와 staleness guard
3. SYNTHETIC_FIXTURES                  합성 자산만으로 회귀 스위트 실행
4. DEPENDENCY_AND_APPROVAL_PREFLIGHT   의존성·장벽 점검(자격증명 이전)
5. SOURCE_FILE_IDENTITY                file id → inventory → 바이트 SHA-256
6. SOURCE_ORACLE_DIFFERENTIAL          등록 build_record 실행 ↔ 어댑터
7. RESULT_GATE                         등록 gate 로 후보 구조 검사(등록 아님)
8. BUNDLE_REPORT                       외부 동결용 digest 와 다음 단계

승인 전에는 1~4 까지만 진행된다. 5 이후는 두 장벽이 모두 열려야 한다.
이 노트북은 SOURCE_MATCH_ORACLE_RECORD 에 아무것도 쓰지 않는다 —
등록은 Codex 인수 뒤 별도 registration PR 의 몫이다.


In [2]:
# 2. ENVIRONMENT — 저장소를 찾아 로드하고, 쓰려는 기능이 실제로 있는지 본다.
#    '경로가 있다' 가 아니라 '필요한 모듈이 거기 있다' 로 판정한다(quest56 과 동일).
import os, sys, json, subprocess

REPO = ''          # 절대경로를 알면 여기에 적는다 (그러면 탐색 생략)
REPO_URL = 'https://github.com/ehdbddl06001-ui/my-github-test'


def _looks_like_repo(path):
    return os.path.isfile(os.path.join(
        path, 'mit-bih', 'q5e_prep_p3_source_match_equivalence.py'))


if not REPO:
    for candidate in ('/content/repo', os.getcwd(),
                      os.path.abspath(os.path.join(os.getcwd(), '..'))):
        if _looks_like_repo(candidate):
            REPO = candidate
            break

if not REPO:
    REPO = '/content/repo'
    if not os.path.isdir(REPO):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO],
                       check=True)
    if not _looks_like_repo(REPO):
        raise RuntimeError('clone 은 됐는데 P3 모듈이 없다 — 브랜치를 확인하라')

sys.path.insert(0, os.path.join(REPO, 'mit-bih'))
import q5d_order_preserving_beat_join as BJ
import q5e_leg2_failure_mechanism_audit as Q5E
import q5e_prep_p1_p2_asset_identity as P12
import q5e_prep_p3_source_match_equivalence as P3

missing = [name for name in P3.module_capabilities() if not hasattr(P3, name)]
if missing:
    raise RuntimeError(f'낡은 사본이다 — 없는 기능: {missing}')

print('repo :', REPO)
print('P3   :', Q5E.sha256_file(P3.__file__))
print('Q5E  :', Q5E.sha256_file(Q5E.__file__))
print('Q5D  :', Q5E.sha256_file(BJ.__file__), '(frozen)')
print()
print(P3.design_card())


repo : /content/repo
P3   : c490f1b99965f0a7e8c61873862c3e8ce7ae55259ae148ba2d8dc71e48b83c49
Q5E  : f96af094ac65c6897c8030d6bfabe65d25cc623bc21b452a932df3bc617c0127
Q5D  : 6b098c67df3c8e2c8c070b093e6e2d801566f548a3173626745c4a126a97f226 (frozen)

EXP-2026-008 / Q5E_PREP_P3_SOURCE_MATCH_EQUIVALENCE - read-only differential, not a result
  spec                 : experiments/specs/EXP-2026-008-q5e-prep-p3-source-match-equivalence.md
  parent spec          : experiments/specs/EXP-2026-008-q5e-leg2-failure-mechanism-audit.md
  registered source    : data.py :: build_record
  source file id       : 1a8mfNbCz5_vPaOWajsX15l93rgEaO_UK
  source folder id     : 1czXZdgSrGttrhOFlNvOHQ3l16ZfluOPX
  source sha256        : 20cde66b01d1172926aa1b84cbb70b70ea28bb20c2e958a2c26bd01d03497ada
  source bytes         : 7744
  adapter fingerprint  : 85c5e2599f8680ca122152fa4daa0936d0393fec2b33048b29f75c81e3895bb5
  oracle harness sha256: c21a5c1d7536a77adcc9167995768689981abfeb6688adf0aa7706e48fa2d873
  oracl

In [3]:
# 3. SYNTHETIC_FIXTURES — 등록 자산을 열기 전에 합성 자산만으로 전 경로를 점검한다.
#    여기서 깨지면 승인을 켜기 전에 고치는 편이 훨씬 싸다. 이 셀은 Drive 를
#    호출하지 않고 등록 data.py 도 열지 않는다 — 합성 producer 로만 돈다.
TEST = os.path.join(REPO, 'mit-bih',
                    'test_q5e_prep_p3_source_match_equivalence.py')
proc = subprocess.run([sys.executable, TEST], capture_output=True, text=True)
print(proc.stdout or '(stdout 없음)')
if proc.returncode != 0:
    print(proc.stderr[-4000:])
    raise RuntimeError(f'합성 회귀 스위트 실패 (exit {proc.returncode})')

print()
print('여섯 required fixture 와 각각이 반증하는 것:')
for name in P3.fixture_names():
    card = P3.fixture_card(name)
    print(' -', name)
    print('     peaks      :', card['peaks'])
    print('     annotations:', card['annotations'])
    print('     반증 대상  :', card['refutes'])
print()
print('required set 일치:',
      list(P3.fixture_names()) == list(Q5E.SOURCE_MATCH_REQUIRED_FIXTURES))
print('fixture 에 정답을 적지 않는다 — 정답은 등록 source 가 말한다.')


114 test functions, 1179 assertions passed


여섯 required fixture 와 각각이 반증하는 것:
 - test_source_match_nearest_already_used_falls_through
     peaks      : [1000, 1012, 1500, 1740, 1980, 2220, 2460, 2700]
     annotations: [[1001, 'N'], [1042, 'V'], [1501, 'A'], [1741, 'L'], [1981, 'J'], [2221, 'R'], [2461, 'a'], [2701, 'e']]
     반증 대상  : that a peak whose nearest annotation is already consumed is dropped.  Peak 1012's nearest is 1001, taken by peak 1000; falling through keeps it against 1042, dropping loses the row entirely
 - test_source_match_distance_tie_goes_to_the_earlier_annotation
     peaks      : [1030, 1500, 1740, 1980, 2220, 2460, 2700]
     annotations: [[1000, 'N'], [1060, 'V'], [1501, 'A'], [1741, 'L'], [1981, 'J'], [2221, 'R'], [2461, 'a'], [2701, 'e']]
     반증 대상  : that an exact distance tie goes to the later annotation.  Peak 1030 is 30 samples from both 1000 and 1060, and the two carry different AAMI classes, so the tie rule is visible in the kept row and not only in 

In [4]:
# 4. DEPENDENCY_AND_APPROVAL_PREFLIGHT — **자격증명 생성 이전** 에 점검한다.
#    의존성 확인을 credential 뒤에 두면, 없는 패키지 때문에 실패하는 실행이
#    이미 토큰을 발급받은 상태가 된다. 순서 자체가 계약이다.
report = P12.check_runtime_dependencies()
print('필요 패키지 :', report['required'])
print('없는 것     :', report['missing'])
print('설치는 하지 않는다 —', report['note'])
print()
print('장벽 1  OPEN_REGISTERED_DATA         :', P3.OPEN_REGISTERED_DATA)
print('장벽 2  EXECUTION_APPROVAL_RECORD    :',
      P3.EXECUTION_APPROVAL_RECORD['granted'])
print('요구 scope                           :', P3.DRIVE_READONLY_SCOPE)
print('거부되는 다른 단계 토큰              :', len(P3.REFUSED_TOKENS), '개')
for note in P3.REFUSED_TOKENS.values():
    print('   -', note.split('.')[0])
print()
# 2026-08-16 사용자가 P3 read-only 실행을 다시 승인했다 — **현재 harness
# (a90d1d2a…) 에 한해서**. 8/15 승인은 kept-row 리더가 harness digest 를
# 바꾸면서 철회됐고, 새 승인은 digest 에 묶여 있다: 앞으로 harness 를 고치면
# 승인이 자동으로 승계되지 않고 문이 다시 닫힌다(P3NotApprovedError 가 두
# digest 를 다 찍어준다). 모듈 기본값 P3.OPEN_REGISTERED_DATA 는 여전히
# False 다 — 여는 것은 이 **호출 지점**뿐이고, 승인 범위 밖(Q5-E 본 실행·
# detect_r·M0~M4·학습·oracle record 등록)으로는 이 두 줄이 아무것도 열지
# 않는다. "한번 돌려보자"고 승인 범위를 넓히지 마라.
APPROVAL = P3.EXECUTION_APPROVAL_TOKEN
OPEN_REGISTERED_DATA = True          # 호출 지점 opt-in (모듈 기본값은 False)
print('APPROVAL 설정 여부 :', P3.execution_is_approved(APPROVAL))
print('실행 승인 상태     :', P3.EXECUTION_APPROVAL_RECORD['granted'])
print('승인이 묶인 harness:',
      P3.EXECUTION_APPROVAL_RECORD['for_oracle_harness_sha256'])
print('이 모듈의 harness  :',
      P3.oracle_harness_identity()['oracle_harness_sha256'],
      '← 위와 같아야 5번 이후가 열린다')
print('모듈 기본값(불변)  :', P3.OPEN_REGISTERED_DATA, '← False 그대로여야 한다')
print(P3.APPROVAL_NOTE)

필요 패키지 : ['google.auth', 'google.colab', 'googleapiclient']
없는 것     : []
설치는 하지 않는다 — install these deliberately and pin them; this preflight never installs a package for you, because a silent upgrade would change the runtime under an identity check

장벽 1  OPEN_REGISTERED_DATA         : False
장벽 2  EXECUTION_APPROVAL_RECORD    : True
요구 scope                           : https://www.googleapis.com/auth/drive.readonly
거부되는 다른 단계 토큰              : 2 개
   - the Q5-E audit execution token
   - the P1/P2 PREP token

APPROVAL 설정 여부 : True
실행 승인 상태     : True
승인이 묶인 harness: c21a5c1d7536a77adcc9167995768689981abfeb6688adf0aa7706e48fa2d873
이 모듈의 harness  : c21a5c1d7536a77adcc9167995768689981abfeb6688adf0aa7706e48fa2d873 ← 위와 같아야 5번 이후가 열린다
모듈 기본값(불변)  : False ← False 그대로여야 한다
Approved (2026-08-16) by the user, for oracle harness a90d1d2a… and no other: the 2026-08-15 approval was withdrawn when the kept-row reader changed the harness digest, and this one is bound to the digest so the next harn

In [5]:
# 5. SOURCE_FILE_IDENTITY — **보고만 한다. 인증하지 않는다.**
#    등록 data.py 는 file id 로만 고른다. 이름이 같은 파일을 찾은 것은 증거가
#    아니다 — 그 치환을 막는 것이 이 PREP 의 존재 이유다.
#    인증·다운로드는 이 셀에 없다. 노트북이 먼저 adapter 를 만들면 terminal
#    guard 가 보지도 못한 채 credential 이 발급되기 때문이다.
print('registered file   :', P3.REGISTERED_SOURCE_NAME, '::',
      P3.REGISTERED_SOURCE_FUNCTION)
print('registered file id:', P3.REGISTERED_SOURCE_FILE_ID)
print('registered folder :', P3.REGISTERED_SOURCE_FOLDER_ID)
print('registered bytes  :', P3.REGISTERED_SOURCE_BYTES)
print('registered sha256 :', P3.REGISTERED_SOURCE_SHA256)
print('ASSETS row        :', P3.REGISTERED_SOURCE_ASSET_ROW)
print()
print('검증 순서: file id 직접 조회 → provider inventory(size/checksum/parents)')
print('          → 바이트 읽기 → 읽은 바이트의 SHA-256 → 그 다음에야 import')
print()
print('gate 순서:')
for index, gate in enumerate(P3.P3_GATE_ORDER, 1):
    print(f'  {index}. {gate}')
print()
print('중단 사유(등록 파일이 아니면 여기서 멈춘다):')
for stop in P3.HARNESS_STOPS:
    print('  -', stop)


registered file   : data.py :: build_record
registered file id: 1a8mfNbCz5_vPaOWajsX15l93rgEaO_UK
registered folder : 1czXZdgSrGttrhOFlNvOHQ3l16ZfluOPX
registered bytes  : 7744
registered sha256 : 20cde66b01d1172926aa1b84cbb70b70ea28bb20c2e958a2c26bd01d03497ada
ASSETS row        : baseline-v10-source

검증 순서: file id 직접 조회 → provider inventory(size/checksum/parents)
          → 바이트 읽기 → 읽은 바이트의 SHA-256 → 그 다음에야 import

gate 순서:
  1. fixture_contract
  2. source_file_id_registered
  3. source_inventory
  4. source_bytes_digest
  5. source_loaded
  6. fixture_differential
  7. candidate_structure

중단 사유(등록 파일이 아니면 여기서 멈춘다):
  - P3_SOURCE_FILE_ID_UNREGISTERED
  - P3_SOURCE_IDENTITY_MISMATCH
  - P3_SOURCE_UNLOADABLE
  - P3_SOURCE_SIGNATURE_UNBINDABLE
  - P3_SOURCE_RUNTIME_ERROR
  - P3_SOURCE_TRACE_UNPROJECTABLE
  - P3_KEPT_ROWS_UNOBSERVABLE
  - P3_FIXTURE_CONTRACT_VIOLATION
  - P3_STUB_SURFACE_INCOMPLETE
  - P3_INJECTED_VALUE_STEERS_MATCHING
  - P3_COLUMNAR_RETURN_UNPROJECTABLE
  - P3_SOURC

In [6]:
# 6. SOURCE_ORACLE_DIFFERENTIAL — 실제 실행 경로. 두 장벽이 모두 열려야 온다.
#    adapter_source=None 이다: 인증과 adapter 생성은 guard **아래**
#    run_p3() 안에서만 일어난다. 노트북은 credential 을 만지지 않는다.
#
#    oracle 은 등록 build_record 자체다. 주입되는 것:
#      - wfdb reader stub (합성 ramp 신호, 실제 ECG 아님)
#      - detect_r stub    (fixture 의 peak, 실제 검출기 미실행)
#      - rr/feature stub  (행이 어느 peak 것인지 스스로 밝히는 행)
#
#    Drive 를 두 가지로 쓴다. 읽기는 **API**(file id 조회 → 다운로드)로 하고
#    guard 아래에서만 인증한다. 마운트는 **출력 bundle 을 쓰기 위해서만**
#    필요하다 — 등록 data.py 를 마운트 경로로 읽지 않는다. 이름이 같은 파일을
#    경로로 집는 것이 이 PREP 가 막으려는 바로 그 치환이기 때문이다.
import os as _os

MOUNT = '/content/drive'
if not _os.path.isdir(_os.path.join(MOUNT, 'MyDrive')):
    from google.colab import drive
    drive.mount(MOUNT)               # 출력 bundle 기록용
print('Drive 마운트 :', _os.path.isdir(_os.path.join(MOUNT, 'MyDrive')))

RESULT = None
OUT_DIR = _os.path.join(MOUNT, 'MyDrive/MedKOS/ecg-model/runs')

# 실행 시각이 이 실행의 식별자이자 출력 폴더 이름이고, 나중에 executed 노트북
# 파일명에도 같은 값을 쓴다. 비워 두면 KST 현재 시각으로 채운다 — 손으로 적던
# 것을 매번 다시 적을 이유가 없다. 특정 값으로 고정하고 싶으면(재현·정정 등)
# 아래 한 줄에 직접 적으면 그 값이 그대로 쓰인다.
from datetime import datetime, timezone, timedelta
TIMESTAMP = ''                        # 예: '20260816T010203'
if not TIMESTAMP:
    TIMESTAMP = datetime.now(timezone(timedelta(hours=9))).strftime(
        '%Y%m%dT%H%M%S')
print('TIMESTAMP :', TIMESTAMP, '— executed 노트북 파일명에도 이 값을 쓴다')

# 형식 검사는 남긴다. 자동 생성이 실패하거나 손으로 이상한 값을 적었을 때,
# 이름 없는/충돌하는 bundle 을 쓰는 것보다 여기서 멈추는 편이 낫다.
import re as _re
if not _re.fullmatch(r'\d{8}T\d{6}', TIMESTAMP):
    raise RuntimeError(
        f'TIMESTAMP 가 YYYYmmddTHHMMSS 형식이 아니다: {TIMESTAMP!r}. 출력 폴더 '
        f'이름이자 이 실행의 식별자이므로, 형식이 어긋나면 진행하지 않는다.')
_os.makedirs(OUT_DIR, exist_ok=True)   # 부모 디렉터리만. bundle 은 run_p3 이 만든다

if P3.execution_is_approved(APPROVAL) and OPEN_REGISTERED_DATA:
    RESULT = P3.run_p3(
        OUT_DIR, timestamp=TIMESTAMP,
        adapter_source=None,          # 인증은 guard 아래에서만
        approval=APPROVAL,
        open_registered_data=OPEN_REGISTERED_DATA,
        file_id=P3.REGISTERED_SOURCE_FILE_ID)
else:
    print('실행하지 않음 — 두 장벽 중 하나 이상이 닫혀 있다.')
    print('  OPEN_REGISTERED_DATA        :', OPEN_REGISTERED_DATA)
    print('  approval token 유효         :', P3.execution_is_approved(APPROVAL))
    print('  EXECUTION_APPROVAL_RECORD   :',
          P3.EXECUTION_APPROVAL_RECORD['granted'])
    print()
    print('승인 없이 여기까지 온 호출은 credential 0회 · Drive API 0회 ·')
    print('등록 바이트 0회 · 출력 디렉터리 생성 0회다.')


Mounted at /content/drive
Drive 마운트 : True
TIMESTAMP : 20260816T161639 — executed 노트북 파일명에도 이 값을 쓴다
Q5-E PREP P3: approval present; 6 registered fixtures; nothing has been opened yet.
read-only execution approval: granted 2026-08-16 by user — read-only execution of EXP-2026-008 Q5-E PREP P3: the candidate adapter against the registered data.py under synthetic dependency injection.
not approved by it: the Q5-E scientific execution, running detect_r() or any real detector, opening a real ECG signal, a V9/V10 cache or any real-record count, M0-M4 aggregation, opening DS2 per-beat labels, opening V10 probabilities, computing association or S PR-AUC, training or retraining any model, registering SOURCE_MATCH_ORACLE_RECORD, modifying, moving or deleting any Drive artifact, correcting the candidate adapter inside this PREP.


Drive scope proven read-only: True


registered data.py verified by id and digest: 20cde66b01d1172926aa1b84cbb70b70ea28bb20c2e958a2c26bd01d03497ada
fixture test_source_match_nearest_already_used_falls_through: equal=False
fixture test_source_match_distance_tie_goes_to_the_earlier_annotation: equal=True
fixture test_source_match_non_aami_symbol_consumes_its_match: equal=False
fixture test_source_match_boundary_cut_consumes_its_match: equal=True
fixture test_source_match_annotation_order_differing_from_sample_order: equal=False
fixture test_source_match_peak_order_change_is_visible: equal=True
status: SOURCE_MATCH_EQUIVALENCE_REQUIRED
bundle committed, payload fold: fa4cfe6cf5f1d7d4a006de812e1fdda8d462bd71431da9abb536fe323ae5b449
SOURCE_MATCH_ORACLE_RECORD is still None: this run registers nothing.


In [7]:
# 7. RESULT_GATE — 판정과 후보 구조 검사. **등록은 하지 않는다.**
#    모든 fixture 가 일치할 때만 후보가 생기고, 그 후보를 현재 main 의
#    Q5E.verify_source_match_equivalence() 에 넣어 구조적으로 통과하는지만 본다.
#    SOURCE_MATCH_ORACLE_RECORD 에는 쓰지 않는다.
if RESULT is None:
    print('결과 없음 — 위 셀이 실행되지 않았다.')
    print('현재 SOURCE_MATCH_ORACLE_RECORD :', Q5E.SOURCE_MATCH_ORACLE_RECORD)
    print('현재 M4.0 sub-gate              :',
          Q5E.verify_source_match_equivalence()['reason'])
else:
    decision = RESULT['decision']
    print('status            :', decision['status'])
    print('harness stop      :', decision['harness_stop'])
    print('first failure     :', decision['first_failure'])
    print('fixtures passed   :', decision['fixtures_passed'], '/',
          decision['fixtures_total'])
    if decision['harness_stop']:
        # 무엇이 막혔는지 여기서 바로 읽는다. 이전 실행에서는 이 값이 bundle
        # 안에만 있어서 Drive 를 열어야 이름을 알 수 있었다 — stop 은 다음
        # 한 번의 실행으로 진단돼야 한다.
        print()
        print('== harness stop detail ==')
        print(decision['detail'])
        # 그리고 왜 막혔는지: 반환값의 모양, stub 호출, 그리고 producer 가
        # 어느 줄에서 무엇을 들고 반환했는지. 이게 없으면 다음 한 번이 또
        # 추측이 된다. 배열 내용은 기록하지 않는다 — 타입과 길이뿐이다.
        stop_context = (RESULT.get('fixture_results') or {}).get(
            'harness_stop_context') or {}
        if stop_context:
            print()
            print('== stop context ==')
            print('fixture            :', stop_context.get('fixture'))
            print('reason             :', stop_context.get('reason'))
            schema = stop_context.get('return_schema') or {}
            print('반환 타입          :', schema.get('type'))
            print('반환 key           :', schema.get('keys'))
            for key, info in sorted((schema.get('columns') or {}).items()):
                print(f'   {key:6s} shape={info.get("shape")} '
                      f'dtype={info.get("dtype")} '
                      f'row_aligned={info.get("row_aligned")}')
            calls = stop_context.get('stub_calls') or {}
            print('stub 호출 횟수     :', calls.get('counts'))
            print('stub 입력 행 수    :', calls.get('input_rows'))
            found = stop_context.get('surface_discovery') or {}
            if found:
                print()
                print('== surface discovery (관측 아님) ==')
                print('아직 선언 안 된 이름 :', found.get('undeclared_names'))
                print('끝까지 갔는가        :', found.get('reached_the_end'))
                if found.get('stopped_with'):
                    print('멈춘 지점            :', found['stopped_with'])
                print('이 pass 의 결과는 관측이 아니다 — 이름 목록만 쓴다.')
            # 투영이 못 가린 경우: producer 자신의 라벨이 무엇으로 이해됐는지.
            # 이게 없으면 '행이 가리지 못했다'는 원인 없는 사실일 뿐이다.
            if stop_context.get('unresolved'):
                print()
                print('== 가리지 못한 것 ==')
                for line in stop_context['unresolved']:
                    print('  -', line)
                print('행 토큰            :', stop_context.get('row_tokens'))
                print('누적 라벨사전      :',
                      stop_context.get('label_dictionary'))
                print('이 fixture 사전    :',
                      stop_context.get('fixture_label_dictionary'))
            trace = stop_context.get('trace') or {}
            if trace:
                print('실행된 함수        :', trace.get('code_name'),
                      f"({trace.get('n_steps')} steps)")
                print('반환한 줄          :', trace.get('returned_from_line'))
                print('실행된 줄          :',
                      trace.get('distinct_lines_executed'))
                print('마지막 locals      :')
                for name, value in sorted(
                        (trace.get('final_locals') or {}).items()):
                    print(f'   {name:16s} = {value}')
        print()
        print('비교는 이뤄지지 않았다. 어댑터와 등록 source 사이의 불일치가')
        print('아니며, 그렇게 보고해서도 안 된다. harness 를 좁게 고치고')
        print('여섯 fixture 를 처음부터 다시 돌린다.')
    print()
    print('| fixture | source digest | adapter digest | equal |')
    print('|---|---|---|---|')
    for entry in (RESULT['differential'] or {}).get('fixtures', ()):
        print(f"| {entry['name']} | {entry['source_result_sha256']} "
              f"| {entry['adapter_result_sha256']} | {entry['equal']} |")
    print()
    gate = RESULT['candidate_gate']
    print('등록 gate 통과(구조) :', gate['ok'])
    for problem in gate['problems']:
        print('   -', problem)
    print('등록 상수에 기록함   :', gate['registered_constant_written'])
    if not decision['equivalence_claimed']:
        print()
        print('불일치가 하나라도 있으면: verdict 는')
        print('SOURCE_MATCH_EQUIVALENCE_REQUIRED 로 남고, real-record count 를')
        print('열지 않으며, 어댑터를 자동 수정하지 않는다. 수정은 별도 PR 이다.')

status            : SOURCE_MATCH_EQUIVALENCE_REQUIRED
harness stop      : False
first failure     : test_source_match_nearest_already_used_falls_through
fixtures passed   : 3 / 6

| fixture | source digest | adapter digest | equal |
|---|---|---|---|
| test_source_match_nearest_already_used_falls_through | 968c064207ef63f25c393b62e2efbd61d234606e3c5c6dfc2c7d256ed94e259c | 84ba34402500c0ee541708bfaec4a1695eaf500001447d39e2f46907af7adf84 | False |
| test_source_match_distance_tie_goes_to_the_earlier_annotation | 8618d216069083fb68281046a0a9bddf15aa72701e03a25fa16585390281a01c | 8618d216069083fb68281046a0a9bddf15aa72701e03a25fa16585390281a01c | True |
| test_source_match_non_aami_symbol_consumes_its_match | 14712d08287bd83453992bfcae7b094e32c36f0694824c81a9a72588c11f00fd | 2bfd7a0c1c3fb712db1285270172b16a45af7684b93a375d0eb47587c7d2a1d5 | False |
| test_source_match_boundary_cut_consumes_its_match | 2fe0cb574169f3918d8b0285e3204692cdf97e575144c7f37c09b1f5438f78cf | 2fe0cb574169f3918d8b028

In [8]:
# 8. BUNDLE_REPORT — 외부 동결용 보고. 이 셀의 **저장된 출력** 이
#    manifest SHA-256 의 외부 anchor 다(번들 안에는 자기 digest 를 적지 않는다).
#    후보 record 의 prep_bundle_sha256 은 이 번들의 payload fold 이므로,
#    후보 자체도 번들 안에 넣지 않고 여기서 보고한다.
if RESULT is None:
    print('보고할 번들이 없다 — 실행 승인 전이다.')
    print('실행되면 이 셀이 다음을 전부 찍는다:')
    for line in ('source file id 와 SHA-256 검증',
                 'adapter fingerprint',
                 'oracle harness SHA-256',
                 'fixture 별 source/adapter digest 와 equal',
                 'required fixture coverage',
                 'first failure',
                 'final verdict',
                 'prep payload fold 전체 64-hex',
                 'manifest SHA-256 전체 64-hex',
                 'Drive run folder 와 folder ID',
                 'SOURCE_MATCH_ORACLE_RECORD registration candidate',
                 '다음 단계가 registration 인지 adapter 수정인지'):
        print('  -', line)
else:
    inventory = RESULT['source_inventory']
    bundle = RESULT['bundle']
    harness = RESULT['harness']
    decision = RESULT['decision']
    print('== source identity ==')
    print('file id                :', inventory['requested_file_id'])
    print('provider sha256        :', inventory.get('provider_sha256'))
    print('읽은 바이트 sha256     :', inventory['observed_sha256'])
    print('등록 digest 와 일치    :', inventory['digest_matches_registered'])
    print()
    print('== identities ==')
    print('adapter fingerprint    :',
          Q5E.source_match_adapter_fingerprint())
    print('oracle harness SHA-256 :', harness['oracle_harness_sha256'])
    print()
    print('== fixtures ==')
    for entry in (RESULT['differential'] or {}).get('fixtures', ()):
        print(f"  {entry['name']}")
        print(f"    source : {entry['source_result_sha256']}")
        print(f"    adapter: {entry['adapter_result_sha256']}")
        print(f"    equal  : {entry['equal']}")
    print('required fixture coverage:',
          sorted(e['name'] for e in
                 (RESULT['differential'] or {}).get('fixtures', ())) ==
          sorted(Q5E.SOURCE_MATCH_REQUIRED_FIXTURES))
    # 주입 helper 가 판정을 흔들지 않았다는 것은 주장하지 않고 매 실행 증명한다:
    # 같은 fixture 를 다른 _z 구현으로 한 번 더 관측해 비교 필드가 하나도
    # 움직이지 않아야 한다. 'untested' 가 섞여 있으면 그 fixture 는 증명되지
    # 않은 것이고, 'violated' 면 애초에 여기까지 오지 않는다.
    print('injected-helper invariance:',
          (RESULT['differential'] or {}).get('stub_invariance_probed'))
    for fixture_name, status in sorted(
            ((RESULT['differential'] or {}).get('stub_invariance') or {}).items()):
        print(f'    {status:10s} {fixture_name}')
    print('first failure           :', decision['first_failure'])
    print('final verdict           :', decision['status'])
    if decision['harness_stop']:
        print('stop detail             :', decision['detail'])
    print()
    print('== bundle ==')
    print('run folder             :', bundle['directory'])
    print('folder ID              : (Drive UI 에서 확인해 ASSETS 에 기록)')
    print('prep payload fold      :', bundle['prep_payload_sha256'])
    print('manifest SHA-256       :',
          bundle['manifest_sha256_freeze_externally'])
    print()
    print('== registration candidate (등록 아님) ==')
    print(json.dumps(RESULT['candidate'], indent=1, sort_keys=True))
    print('SOURCE_MATCH_ORACLE_RECORD 현재값 :',
          Q5E.SOURCE_MATCH_ORACLE_RECORD)
    print()
    print('다음 단계:', decision['next_step'])


== source identity ==
file id                : 1a8mfNbCz5_vPaOWajsX15l93rgEaO_UK
provider sha256        : 20cde66b01d1172926aa1b84cbb70b70ea28bb20c2e958a2c26bd01d03497ada
읽은 바이트 sha256     : 20cde66b01d1172926aa1b84cbb70b70ea28bb20c2e958a2c26bd01d03497ada
등록 digest 와 일치    : True

== identities ==
adapter fingerprint    : 85c5e2599f8680ca122152fa4daa0936d0393fec2b33048b29f75c81e3895bb5
oracle harness SHA-256 : c21a5c1d7536a77adcc9167995768689981abfeb6688adf0aa7706e48fa2d873

== fixtures ==
  test_source_match_nearest_already_used_falls_through
    source : 968c064207ef63f25c393b62e2efbd61d234606e3c5c6dfc2c7d256ed94e259c
    adapter: 84ba34402500c0ee541708bfaec4a1695eaf500001447d39e2f46907af7adf84
    equal  : False
  test_source_match_distance_tie_goes_to_the_earlier_annotation
    source : 8618d216069083fb68281046a0a9bddf15aa72701e03a25fa16585390281a01c
    adapter: 8618d216069083fb68281046a0a9bddf15aa72701e03a25fa16585390281a01c
    equal  : True
  test_source_match_non_aami_symbol_c

## 실행 후 절차

1. 이 노트북을 **출력을 담아** `notebooks/executed/` 에 저장한다. 셀 8 의
   저장된 출력이 manifest SHA-256 의 외부 anchor 이고, 후보 record 가 남는
   유일한 곳이다.
2. Drive run folder 와 folder ID 를 `research/ASSETS.md` 에 등록한다.
3. Codex 가 번들을 읽기 전용으로 다시 가져와 결과를 인수한다.
4. 인수된 뒤에야 **별도 registration PR** 이
   `SOURCE_MATCH_ORACLE_RECORD` 를 채운다. 이 노트북은 채우지 않는다.

불일치가 하나라도 있었다면 3~4 대신: 불일치 trace 와 두 digest 를 보존한 채
Codex 재검토로 넘기고, 어댑터 정정은 별도 PR 에서 하고 전 fixture 를 처음부터
다시 돌린다. **PASS 를 주장하지 않는다.**
